# NVFP4 Model serving with vllm on Turing T4 in Google Colab with Nemotron Nano 9B

This notebook is a proof of concept for running NVFP4 model checkpoints on T4 with vllm.  To limit runtime, it uses the public T4 wheelhouse pre-built wheels and `nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4`.  Unlike later architecture levels, this architecture level uses FP16 for activations.

The model download starts early in the background and logs explicit
`MODEL_DOWNLOAD_COMPLETE`, `MODEL_DOWNLOAD_FAILED`, or
`MODEL_DOWNLOAD_ABORTED` markers. Setup cells continue while the model
is downloading. vLLM startup is not wall-clock capped. Individual HTTP
probes are capped so a single `curl` cannot hang.

The MODEL is defined in this cell as variable -- substitute your own model here by specifying the HF path.  The free T4 Google colab is limited to a T4 with 12 GB device memory which must hold the model, activations, and kernels. Pick a larger (non-free) device to run larger models.

## Trying other checkpoints

The model is set in one place, the environment block in the next cell, and any NVFP4 checkpoint
vLLM can load will work. Two alternatives are commented there: `nvidia/NVIDIA-Nemotron-3-Nano-4B-NVFP4`,
which is the same architecture family and quicker on a T4, and `mgoin/Qwen3-0.6B-NVFP4`, which is
smaller still and has no recurrent layers. The Qwen one is a compressed-tensors checkpoint and
declares its own quantization, so it needs `MODEL_QUANT=compressed-tensors`; the Nemotron
checkpoints carry no `quantization_config` and need `modelopt_mixed` named explicitly.

Nothing here is tuned for a particular checkpoint. What limits the choice is the T4's 16 GB and the
free Colab session, not the format.


In [ ]:
%%bash
set -euxo pipefail

export DEBIAN_FRONTEND=noninteractive
export PIP_BREAK_SYSTEM_PACKAGES=1
export PYTHONNOUSERSITE=1
export PYTHONUNBUFFERED=1

cat > /content/nvfp4_env.sh <<'EOF'
export DEBIAN_FRONTEND=noninteractive
export PIP_BREAK_SYSTEM_PACKAGES=1
export PYTHONNOUSERSITE=1
export PYTHONUNBUFFERED=1
export PYTHONFAULTHANDLER=1
export HF_HUB_ENABLE_HF_TRANSFER=1
export CUDA_HOME=/usr/local/cuda
export PATH="${CUDA_HOME}/bin:${PATH}"
# Any NVFP4 checkpoint vLLM can load will do. Set these four together and rerun
# from this cell. MODEL_QUANT is needed because a checkpoint may or may not
# declare its own quantization: the Nemotron ones carry no quantization_config,
# so it has to be named, while compressed-tensors checkpoints declare theirs.
export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
export MODEL_DIR=/content/models/nemotron-nano-9b-v2-nvfp4
export MODEL_QUANT=modelopt_mixed

# Smaller and quicker on a T4, same architecture family:
# export MODEL_REPO=nvidia/NVIDIA-Nemotron-3-Nano-4B-NVFP4
# export MODEL_NAME=nemotron-3-nano-4b-nvfp4-sanity
# export MODEL_DIR=/content/models/nemotron-3-nano-4b-nvfp4
# export MODEL_QUANT=modelopt_mixed

# Smallest, and no recurrent layers, so it needs no mamba wheels:
# export MODEL_REPO=mgoin/Qwen3-0.6B-NVFP4
# export MODEL_NAME=qwen3-0.6b-nvfp4-sanity
# export MODEL_DIR=/content/models/qwen3-0.6b-nvfp4
# export MODEL_QUANT=compressed-tensors
export WHEELHOUSE_ARCHIVE=/content/nvfp4_t4_wheelhouse.tgz
export WHEELHOUSE_DIR=/content/nvfp4_t4_wheelhouse
export WHEEL_DIR=/content/nvfp4_t4_wheelhouse/wheels
export LOG_DIR=/content/nvfp4_logs
export MODEL_DOWNLOAD_LOG=/content/nvfp4_logs/model_download.log
export SERVER_LOG=/content/nvfp4_logs/vllm_server_triton_attn.log
export REQUEST_LOG=/content/nvfp4_logs/request.log
EOF

. /content/nvfp4_env.sh
mkdir -p "${LOG_DIR}"

printf 'MODEL_REPO=%s\n' "${MODEL_REPO}"
printf 'MODEL_NAME=%s\n' "${MODEL_NAME}"
printf 'MODEL_DIR=%s\n' "${MODEL_DIR}"
printf 'LOG_DIR=%s\n' "${LOG_DIR}"
python3 --version
free -h
df -h /content
nvidia-smi || true


MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
MODEL_DIR=/content/models/nemotron-nano-9b-v2-nvfp4
LOG_DIR=/content/nvfp4_logs
Python 3.12.13
               total        used        free      shared  buff/cache   available
Mem:            12Gi       708Mi       8.5Gi       2.0Mi       3.4Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   48G   66G  42% /
Tue Jun 30 02:57:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                     

+ export DEBIAN_FRONTEND=noninteractive
+ DEBIAN_FRONTEND=noninteractive
+ export PIP_BREAK_SYSTEM_PACKAGES=1
+ PIP_BREAK_SYSTEM_PACKAGES=1
+ export PYTHONNOUSERSITE=1
+ PYTHONNOUSERSITE=1
+ export PYTHONUNBUFFERED=1
+ PYTHONUNBUFFERED=1
+ cat
+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/n

Start model download in background.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh
mkdir -p "${LOG_DIR}"
mkdir -p "$(dirname "${MODEL_DIR}")"

python3 -m pip install --upgrade pip
python3 -m pip install \
  huggingface_hub \
  hf_transfer

pid_file=/content/nvfp4_model_download.pid
if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}" \
  2>/dev/null; then
  echo "model download already complete"
  tail -80 "${MODEL_DOWNLOAD_LOG}" || true
  exit 0
fi

if [ -f "${pid_file}" ]; then
  old_pid="$(cat "${pid_file}")"
  if ps -p "${old_pid}" >/dev/null 2>&1; then
    echo "model download already running: ${old_pid}"
    tail -80 "${MODEL_DOWNLOAD_LOG}" || true
    exit 0
  fi
fi

: > "${MODEL_DOWNLOAD_LOG}"

(
  set -uxo pipefail
  . /content/nvfp4_env.sh
  trap 'date; echo MODEL_DOWNLOAD_ABORTED; exit 130' HUP INT TERM
  date
  hf download \
    "${MODEL_REPO}" \
    --repo-type model \
    --local-dir "${MODEL_DIR}"
  rc=$?
  date
  echo "hf download exit code: ${rc}"
  if [ "${rc}" = 0 ]; then
    echo MODEL_DOWNLOAD_COMPLETE
  else
    echo MODEL_DOWNLOAD_FAILED
  fi
  exit "${rc}"
) > "${MODEL_DOWNLOAD_LOG}" 2>&1 &

download_pid=$!
echo "${download_pid}" > "${pid_file}"
echo "download_pid=${download_pid}"
echo "model download log: ${MODEL_DOWNLOAD_LOG}"
sleep 2
ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
  -p "${download_pid}" || true
tail -80 "${MODEL_DOWNLOAD_LOG}" || true


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 15.1 MB/s  0:00:00
download_pid=694
model download log: /content/nvfp4_logs/model_download.log
    PID    PPID STAT     ELAPSED %MEM %CPU CMD
    694     622 S          00:02  0.0  0.0 bash
+ set -uxo pipefail
+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/usr/local/cuda/bin

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Install libraries.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

apt-get update
apt_pkgs=(
  ca-certificates
  curl
  git
  libxcb1
  python3-dev
  python3.12-dev
)
apt-get install -y --no-install-recommends "${apt_pkgs[@]}"

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,797 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Install pytorch, transformers, modelopt, etc.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

python3 -m pip install 'setuptools<82' wheel

need_torch=1
if python3 - <<'PY'
import sys
try:
    import torch
    import torchvision
    import torchaudio
except Exception:
    raise SystemExit(1)

ok = (
    torch.__version__ == "2.11.0+cu130"
    and torchvision.__version__ == "0.26.0+cu130"
    and torchaudio.__version__ == "2.11.0+cu130"
)
raise SystemExit(0 if ok else 1)
PY
then
  need_torch=0
fi

if [ "${need_torch}" = 1 ]; then
  torch_index=https://download.pytorch.org/whl/cu130
  python3 -m pip install \
    'torch==2.11.0+cu130' \
    'torchvision==0.26.0+cu130' \
    'torchaudio==2.11.0+cu130' \
    --index-url "${torch_index}"
else
  echo "torch packages already match cu130 targets"
fi

python3 -m pip install \
  'numpy<2.4' \
  safetensors \
  tqdm \
  'mistral_common>=1.10.0' \
  'transformers==5.8.0' \
  'nvidia-modelopt==0.43.0'

python3 - <<'PY'
import torch
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
    print("capability", torch.cuda.get_device_capability(0))
PY

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 133.1 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 218.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 37.6 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 228.2 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 67.5 MB/s  0:00:04
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 145.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 153.5 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 360.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 300.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 233.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 65.8 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 173.0 

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Define library paths and save them to file.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

python3 - <<'PY' > /content/nvfp4_cuda_libs.sh
from pathlib import Path
import site

dirs = []
for root in site.getsitepackages():
    base = Path(root)
    torch_lib = base / "torch" / "lib"
    if torch_lib.is_dir():
        dirs.append(str(torch_lib))
    nvidia_root = base / "nvidia"
    if nvidia_root.is_dir():
        for lib_dir in sorted(nvidia_root.glob("*/lib")):
            dirs.append(str(lib_dir))

lib_path = ":".join(dict.fromkeys(dirs))
print(f"export NVFP4_LIB_PATH={lib_path!r}")
print('export LD_LIBRARY_PATH="${NVFP4_LIB_PATH}:${LD_LIBRARY_PATH:-}"')
PY

cat /content/nvfp4_cuda_libs.sh
cat /content/nvfp4_cuda_libs.sh >> /content/nvfp4_env.sh
. /content/nvfp4_env.sh

python3 - <<'PY'
import torch
print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
PY


export NVFP4_LIB_PATH='/usr/local/lib/python3.12/dist-packages/torch/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cccl/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cupti/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib:/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib:/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib:/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib:/usr/local/lib/python3.12/dist-pack

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Install prebuilt vllm, mamba, etc.  (We can also build locally.  We install prebuilt binaries because build time exceeds the free T4 allocation in Google colab.)

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

WHEELHOUSE_BASE=https://huggingface.co/datasets/mgschwind
WHEELHOUSE_REPO=fleetwide-nvfp4-t4-wheelhouse
WHEELHOUSE_KIND=resolve/main
WHEELHOUSE_FILE=fleetwide_nvfp4_t4_wheelhouse.tgz
WHEELHOUSE_URL="${WHEELHOUSE_BASE}/${WHEELHOUSE_REPO}"
WHEELHOUSE_URL="${WHEELHOUSE_URL}/${WHEELHOUSE_KIND}"
WHEELHOUSE_URL="${WHEELHOUSE_URL}/${WHEELHOUSE_FILE}"
WHEELHOUSE_SHA256=\
40d4493dbd12d251c01f24ddff9b9a5b2b01b680ed2c22fc1c3604caa04a3696

archive_ok=0
if [ -f "${WHEELHOUSE_ARCHIVE}" ]; then
  if echo "${WHEELHOUSE_SHA256}  ${WHEELHOUSE_ARCHIVE}" \
    | sha256sum -c -; then
    archive_ok=1
  fi
fi

if [ "${archive_ok}" = 0 ]; then
  curl \
    -L \
    --fail \
    --retry 5 \
    --retry-delay 5 \
    --connect-timeout 10 \
    --max-time 900 \
    "${WHEELHOUSE_URL}" \
    -o "${WHEELHOUSE_ARCHIVE}"
fi

echo "${WHEELHOUSE_SHA256}  ${WHEELHOUSE_ARCHIVE}" | sha256sum -c -
rm -rf "${WHEELHOUSE_DIR}"
mkdir -p "${WHEELHOUSE_DIR}"
tar -xzf "${WHEELHOUSE_ARCHIVE}" -C "${WHEELHOUSE_DIR}"
find "${WHEEL_DIR}" -maxdepth 1 -type f -name '*.whl' -print

du -sh "${WHEELHOUSE_DIR}"
free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


/content/nvfp4_t4_wheelhouse.tgz: OK
/content/nvfp4_t4_wheelhouse/wheels/vllm-0.1.dev17315+g5afe87873.d20260606-cp312-cp312-linux_x86_64.whl
/content/nvfp4_t4_wheelhouse/wheels/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl
/content/nvfp4_t4_wheelhouse/wheels/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl
144M	/content/nvfp4_t4_wheelhouse
               total        used        free      shared  buff/cache   available
Mem:            12Gi       973Mi       1.7Gi       3.0Mi        10Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   60G   54G  53% /
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ export MODEL_DIR=/content/models/nemotron-nano-9b-v2-nvfp4
++ MODEL_DIR=/content/models/nemotron-nano-9b-v2-nvfp4
++ export WHEELHOUSE_ARCHIVE=/content/nvfp4_t4_wheelhouse.tgz
++ WHEELHOUSE_ARCHIVE=/conten

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Pip install mamba and confirm successful vllm installation.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

python3 -m pip install \
  --find-links "${WHEEL_DIR}" \
  'causal-conv1d==1.6.1' \
  'mamba-ssm==2.3.1'

vllm_wheel="$(find "${WHEEL_DIR}" -name 'vllm*.whl' -print -quit)"
need_vllm=1
if python3 - <<'PY'
try:
    import vllm
except Exception:
    raise SystemExit(1)
raise SystemExit(0 if "g5afe87873" in vllm.__version__ else 1)
PY
then
  need_vllm=0
fi

if [ "${need_vllm}" = 1 ]; then
  python3 -m pip install "${vllm_wheel}"
else
  echo "expected vLLM wheel already installed"
fi

python3 - <<'PY'
import torch
import vllm
import causal_conv1d
import mamba_ssm

print("torch", torch.__version__)
print("torch cuda", torch.version.cuda)
print("vllm", vllm.__version__)
print("causal_conv1d", causal_conv1d.__file__)
print("mamba_ssm", mamba_ssm.__file__)
PY

free -h
df -h /content
tail -40 "${MODEL_DOWNLOAD_LOG}" || true


Looking in links: /content/nvfp4_t4_wheelhouse/wheels
Processing ./nvfp4_t4_wheelhouse/wheels/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl
Processing ./nvfp4_t4_wheelhouse/wheels/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl

Processing ./nvfp4_t4_wheelhouse/wheels/vllm-0.1.dev17315+g5afe87873.d20260606-cp312-cp312-linux_x86_64.whl
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cuda-tile[tileiras] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 50.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 44.3 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

Confirm model download has finished.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

pid_file=/content/nvfp4_model_download.pid
if [ ! -f "${pid_file}" ]; then
  echo "missing download pid file: ${pid_file}"
  exit 1
fi

download_pid="$(cat "${pid_file}")"
echo "download_pid=${download_pid}"
echo "model download log: ${MODEL_DOWNLOAD_LOG}"

if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}"; then
  tail -120 "${MODEL_DOWNLOAD_LOG}" || true
else
  tail -n +1 --pid="${download_pid}" -F "${MODEL_DOWNLOAD_LOG}" |
    sed \
      -e '/MODEL_DOWNLOAD_COMPLETE/q' \
      -e '/MODEL_DOWNLOAD_FAILED/q' \
      -e '/MODEL_DOWNLOAD_ABORTED/q'
fi

if grep -q MODEL_DOWNLOAD_COMPLETE "${MODEL_DOWNLOAD_LOG}"; then
  echo MODEL_DOWNLOAD_COMPLETE
else
  echo "model download did not complete successfully"
  exit 1
fi

du -sh "${MODEL_DIR}"
find "${MODEL_DIR}" -maxdepth 1 -type f -print
free -h
df -h /content
nvidia-smi || true


download_pid=694
model download log: /content/nvfp4_logs/model_download.log
+ set -uxo pipefail
+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_RE

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

## Optional Offline Diagnostic

Run this cell if you want to verify model load and one generation without HTTP serving. Skip it for the fastest path to `vllm serve`. It still has to initialize vLLM and may take a long time on T4.

Ass sanity check we call `llm.generate` with a sanity test question. (`llm.generate(["What is the Capital of Austria? Answer with exactly one word."], params)`).  The correct answer `Vienna` is interspersed at the end with the diagnostic output.

In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

LLM_OBJECT_LOG=/content/nvfp4_logs/llm_object_triton_attn.log
: > "${LLM_OBJECT_LOG}"

export CUDA_VISIBLE_DEVICES=0
export VLLM_LOGGING_LEVEL=DEBUG
export VLLM_LOG_STATS_INTERVAL=1
export PYTHONFAULTHANDLER=1
export TORCH_SHOW_CPP_STACKTRACES=1
export NCCL_DEBUG=INFO
export VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0

python3 -u - <<'PY' 2>&1 | tee -a "${LLM_OBJECT_LOG}"
import faulthandler
import os
import sys
import time

faulthandler.enable()
faulthandler.dump_traceback_later(120, repeat=True, file=sys.stderr)

print("STEP import vllm begin", flush=True)
from vllm import LLM, SamplingParams
print("STEP import vllm done", flush=True)

model = "/content/models/nemotron-nano-9b-v2-nvfp4"

print("STEP construct LLM begin", flush=True)
t0 = time.time()
llm = LLM(
    model=model,
    tokenizer=model,
    quantization=os.environ.get("MODEL_QUANT", "modelopt_mixed"),
    tensor_parallel_size=1,
    trust_remote_code=True,
    dtype="float16",
    max_model_len=512,
    max_num_batched_tokens=512,
    max_num_seqs=1,
    gpu_memory_utilization=0.88,
    cpu_offload_gb=0,
    attention_backend="TRITON_ATTN",
    linear_backend="marlin",
)
print("STEP construct LLM done", round(time.time() - t0, 2), flush=True)

params = SamplingParams(max_tokens=1024, temperature=0)
print("STEP generate begin", flush=True)
t0 = time.time()
out = llm.generate(["What is the Capital of Austria? Answer with exactly one word."], params)
print("STEP generate done", round(time.time() - t0, 2), flush=True)
print(out[0].outputs[0].text, flush=True)
PY

printf 'offline LLM log: %s\n' "${LLM_OBJECT_LOG}"


STEP import vllm begin
INFO 06-30 03:04:21 [__init__.py:213] Installed in-memory Triton fp8e4nv pre-SM89 patch
DEBUG 06-30 03:04:22 [plugins/__init__.py:36] No plugins for group vllm.platform_plugins found.
DEBUG 06-30 03:04:22 [platforms/__init__.py:37] Checking if TPU platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.py:56] TPU platform is not available because: No module named 'libtpu'
DEBUG 06-30 03:04:22 [platforms/__init__.py:62] Checking if CUDA platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.py:85] Confirmed CUDA platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.py:113] Checking if ROCm platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.py:127] ROCm platform is not available because: No module named 'amdsmi'
DEBUG 06-30 03:04:22 [platforms/__init__.py:134] Checking if XPU platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.py:165] Checking if CPU platform is available.
DEBUG 06-30 03:04:22 [platforms/__init__.

+ . /content/nvfp4_env.sh
++ export DEBIAN_FRONTEND=noninteractive
++ DEBIAN_FRONTEND=noninteractive
++ export PIP_BREAK_SYSTEM_PACKAGES=1
++ PIP_BREAK_SYSTEM_PACKAGES=1
++ export PYTHONNOUSERSITE=1
++ PYTHONNOUSERSITE=1
++ export PYTHONUNBUFFERED=1
++ PYTHONUNBUFFERED=1
++ export PYTHONFAULTHANDLER=1
++ PYTHONFAULTHANDLER=1
++ export HF_HUB_ENABLE_HF_TRANSFER=1
++ HF_HUB_ENABLE_HF_TRANSFER=1
++ export CUDA_HOME=/usr/local/cuda
++ CUDA_HOME=/usr/local/cuda
++ export PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ PATH=/usr/local/cuda/bin:/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
++ export MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ MODEL_REPO=nvidia/NVIDIA-Nemotron-Nano-9B-v2-NVFP4
++ export MODEL_NAME=nemotron-nano-9b-v2-nvfp4-sanity
++ MODEL_NAME=nemotron-nano-9b-v2-nvfp4

## Optional vllm serve test

Run the following two cells if you want to test vllm model load and one generation with HTTP serving.  This test has to initialize both vLLM and the HTTP service and will take even longer than the offline diagnostic.


In [ ]:
%%bash
set -euxo pipefail
. /content/nvfp4_env.sh

export CUDA_VISIBLE_DEVICES=0
export VLLM_LOGGING_LEVEL=DEBUG
export VLLM_LOG_STATS_INTERVAL=1
export PYTHONFAULTHANDLER=1
export TORCH_SHOW_CPP_STACKTRACES=1
export NCCL_DEBUG=INFO
export VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0

: > "${SERVER_LOG}"
free -h
df -h /content
nvidia-smi || true

echo "vllm log: ${SERVER_LOG}"
tail -n +1 -F "${SERVER_LOG}" |
  sed '/NVFP4_HEALTH_READY/q' &
log_tail_pid=$!

vllm serve "${MODEL_DIR}" \
  --served-model-name "${MODEL_NAME}" \
  --tokenizer "${MODEL_DIR}" \
  --quantization "${MODEL_QUANT}" \
  --tensor-parallel-size 1 \
  --trust-remote-code \
  --dtype float16 \
  --max-model-len 512 \
  --max-num-batched-tokens 512 \
  --max-num-seqs 1 \
  --gpu-memory-utilization 0.88 \
  --cpu-offload-gb 0 \
  --attention-backend TRITON_ATTN \
  --linear-backend marlin \
  --uvicorn-log-level debug \
  --enable-log-requests \
  --enable-log-outputs \
  --log-error-stack \
  --host 127.0.0.1 \
  --port 8000 \
  > "${SERVER_LOG}" 2>&1 &

server_pid=$!
echo "${server_pid}" > /content/nvfp4_vllm_server.pid
echo "server_pid=${server_pid}"

ready=0
poll=1
while [ "${ready}" = 0 ]; do
  echo "health poll ${poll}"
  date
  ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
    -p "${server_pid}" || true
  free -h
  nvidia-smi || true

  if curl \
    --fail-with-body \
    --connect-timeout 2 \
    --max-time 5 \
    http://127.0.0.1:8000/health; then
    ready=1
    break
  fi

  if ! ps -p "${server_pid}" >/dev/null 2>&1; then
    echo "vllm server exited before health became ready"
    wait "${server_pid}"
  fi

  poll=$((poll + 1))
  sleep 10
done

echo NVFP4_HEALTH_READY >> "${SERVER_LOG}"
wait "${log_tail_pid}" || true

echo "server ready on http://127.0.0.1:8000"
echo "vllm log: ${SERVER_LOG}"
tail -200 "${SERVER_LOG}" || true


Use `curl` to sanity test the model:

In [ ]:
%%bash
set -uxo pipefail
. /content/nvfp4_env.sh

: > "${REQUEST_LOG}"

curl \
  --fail-with-body \
  --connect-timeout 5 \
  --max-time 300 \
  -X POST \
  http://127.0.0.1:8000/v1/chat/completions \
  -H 'Content-Type: application/json' \
  --data-binary @- <<JSON 2>&1 | tee -a "${REQUEST_LOG}"
{
  "model": "${MODEL_NAME}",
  "messages": [
    {
      "role": "system",
      "content": "/no_think"
    },
    {
      "role": "user",
      "content": "What is the Capital of Austria? Answer with exactly one word."
    }
  ],
  "max_tokens": 8,
  "temperature": 0
}
JSON
curl_rc=${PIPESTATUS[0]}

echo "curl_rc=${curl_rc}"
printf 'request log: %s\n' "${REQUEST_LOG}"
printf 'vllm log: %s\n' "${SERVER_LOG}"
cat "${REQUEST_LOG}" || true
ps -o pid,ppid,stat,etime,%mem,%cpu,cmd \
  -p "$(cat /content/nvfp4_vllm_server.pid)" || true
nvidia-smi || true
echo "--- vllm log tail 400 ---"
tail -400 "${SERVER_LOG}" || true
echo "--- end vllm log tail 400 ---"
exit "${curl_rc}"
